# Phase 4c: H2a/H2b Mediation — tech_share, hma, ln_ea

**Date:** 2026-11-03  
**Driver:** `code/R/04c_step2_mediation.R --date 2026-11-03`  
**Results:** `results/2026-11-03/h2/`

---

## What is mediation analysis?

A mediation test asks: does AI data-center activity affect renewable energy *through* an intermediate variable M, or is the effect entirely direct?

The Baron-Kenny three-step framework tests this with three equations:

```
(a) M  = b1·AI(t-1) + X + FE          ← Does AI predict the mediator?
(b) Y  = c1·M      + X + FE           ← Does the mediator predict the outcome?
(c) Y  = e1·M + e2·AI(t-1) + X + FE  ← Is the M→Y link still there when AI is included?
```

The **indirect effect** (the part of AI's total effect that travels through M) is `b1 × e1`.

The **channel holds** if:
- b1 is significant (AI predicts M)
- c1 is significant (M predicts Y)
- e1 is significant (M→Y link survives conditioning on AI)

We report the indirect effect three ways:
1. **Sobel test:** `z = b1·e1 / sqrt(e1²·SE_b1² + b1²·SE_e1²)` — fast, assumes normality
2. **Cluster bootstrap CI:** 999 reps, resample by state, seed=42 — more reliable with panel data
3. **Share mediated:** `b1·e1 / a1`, where a1 is the total AI effect from Step 1 (H1)

**Controls (X):** ctrl4 = ln_gdp_pc + manuf_share + price_c_kwh + pop_growth (H1 baseline).  
**FE:** state + year.  
**SEs:** clustered by state (CRV1).  
**Sample:** 50 states, 2015–2025, N ≈ 550 (lag complete cases).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

DATE = '2026-11-03'
RESULTS = f'../results/{DATE}/h2/'
TABLES  = f'../results/{DATE}/tables/'
FIGS    = f'../results/{DATE}/figures/'

h2 = pd.read_csv(RESULTS + 'h2_results.csv')
print(f'Rows: {len(h2)}')
print(h2['mediator'].value_counts())
print(h2['equation'].value_counts())

## Mediator 1: tech_share (main H2a)

**M = NAICS 5415 share of covered employment, mean-centered.**

H2a posits: AI data centers attract technology workers → states with more tech workers build more renewables.

The test: does lagged AI significantly predict tech_share after controlling for state+year FE and ctrl4?

Note: tech_share was **rejected** as an H1 control because it is this mediator — controlling for it in H1 would remove the AI→tech_share→outcome pathway (bad control problem).

In [ ]:
# Load tech_share table
ts = pd.read_csv(TABLES + 'h2_tech_share__mean_centered_.csv')
display(ts.set_index('Statistic'))

In [ ]:
from IPython.display import Image
Image(FIGS + 'h2_path_tech_share_c.png')

### Reading (tech_share)

**First stage fails.** AI(t-1) does not significantly predict tech_share after state+year FE (b1 = −0.000157, p = 0.454 for all three outcomes). The first-stage failure means the channel cannot hold regardless of downstream significance.

All Sobel p-values are 0.5–0.8 and all bootstrap CIs span zero.

**Interpretation:** Within-state variation in tech worker share is not driven by AI data-center concentration. This is consistent with (1) the small within-state SD of ai_dc_per1000 (≈0.57), and (2) state FE absorbing time-invariant tech agglomeration. The H2a tech-worker channel **is not supported** in these data.

## Mediator 2: hma (memo version)

**M = HMA (High-tech Manufacturing Activity coupling index, built as AI × tech_share).**

⚠️ **Mechanical caution:** HMA is constructed as AI × tech_share, so regressing HMA on AI(t-1) will produce a positive b1 by construction — this tells us nothing about whether the channel holds. The first-stage coefficient is uninformative as evidence.

Run per the methods memo for completeness; results should be interpreted with this caveat.

In [ ]:
hma = pd.read_csv(TABLES + 'h2_hma.csv')
display(hma.set_index('Statistic'))

In [ ]:
Image(FIGS + 'h2_path_hma.png')

### Reading (hma)

As expected, b1 = 0.042 (p < 0.001) — mechanically positive since HMA = AI × tech_share.

However, **all downstream equations (c1, e1) are non-significant** for every outcome. The HMA → Y path does not hold. Sobel p-values: 0.522–0.991; all bootstrap CIs span zero.

**Conclusion:** The HMA memo channel is not supported. Even setting aside the mechanical first stage, HMA has no detectable association with renewable outcomes after controlling for state+year FE.

## Mediator 3: ln_ea (H2b — economic density)

**M = ln(nonfarm GDP per square mile).**

H2b posits: AI data centers increase economic density → denser economies adopt more renewables.

⚠️ **Redundancy caveat (EXPERIMENTS.md §0 item 9):** ln_ea is nearly redundant under state FE. Land area is constant within states, so within-state variation in economic density is entirely driven by nonfarm GDP growth — which is already captured by ln_gdp_pc and pop_growth in ctrl4. We report the within-state correlation to quantify this.

We also check whether AI significantly predicts ln_ea after the controls absorb the GDP growth path.

In [ ]:
lnea = pd.read_csv(TABLES + 'h2_ln_ea.csv')
display(lnea.set_index('Statistic'))

# Redundancy check
redund = pd.read_csv(RESULTS + 'ln_ea_redundancy.csv')
print('\nln_ea redundancy (within-state correlations):')
display(redund)

In [ ]:
Image(FIGS + 'h2_path_ln_ea.png')

### Reading (ln_ea)

**First stage fails.** AI(t-1) does not significantly predict ln_ea (b1 = −0.003, p = 0.296) once state+year FE and ctrl4 controls are included. The redundancy check confirms why: within-state ln_ea has very high correlation with ln_gdp_pc, so ln_gdp_pc already absorbs the AI→EA pathway.

Notably, for ln_ec and ln_ee, the downstream equations (b) and (c) are significant (c1 p ≈ 0.02–0.007, e1 p ≈ 0.02–0.006). Economic density does predict energy outcomes — but AI doesn't predict economic density, so the channel is blocked at the first stage.

**Conclusion:** The H2b economic-density channel is not supported. The ln_ea → Y path exists but AI cannot be shown to cause it.

## Summary: Mediation Results

| Mediator | First stage (b1 p) | Downstream | Channel? | Note |
|---|---|---|---|---|
| tech_share (main H2a) | 0.454 | n/s | **No** | First stage fails for all 3 outcomes |
| hma (memo) | < 0.001 (mechanical) | n/s | **No** | b1>0 by construction; c1/e1 n/s |
| ln_ea (H2b) | 0.296 | mixed (ln_ec, ln_ee significant) | **No** | First stage fails; ln_gdp_pc absorbs |

**Overall finding:** Neither the tech-worker channel (H2a) nor the economic-density channel (H2b) is supported. The null finding is consistent with the small within-state SD of AI (≈0.57) and the fact that state FE and GDP controls absorb most of the variation that would drive these mediators.

This is a **negative result for mediation**, not a negative result for H1. The H1 total effect (new_renew_mw: +70.8 MW, p=0.033) stands; we simply cannot decompose it through these two channels with the available panel variation.

**Next step:** Phase 4d — spatial analysis (H3: Moran's I, SLX, SDM with direct/indirect impacts).

In [ ]:
# All indirect effects: summary plot
indirect_rows = h2[h2['equation'] == 'indirect'].copy()
sobel = indirect_rows[indirect_rows['vcov_type'] == 'Sobel']

fig, ax = plt.subplots(figsize=(8, 5))
colors = {'tech_share_c': 'steelblue', 'hma': 'darkorange', 'ln_ea': 'seagreen'}
labels = {'tech_share_c': 'tech_share (H2a main)',
          'hma': 'hma (H2a memo, mech.)',
          'ln_ea': 'ln_ea (H2b)'}
outcomes_plot = ['ln_ec', 'ln_ee', 'renew_share']
y_positions = {'ln_ec': 2, 'ln_ee': 1, 'renew_share': 0}
offsets = {'tech_share_c': -0.2, 'hma': 0.0, 'ln_ea': 0.2}

for med in ['tech_share_c', 'hma', 'ln_ea']:
    for Y in outcomes_plot:
        row = sobel[(sobel['mediator'] == med) & (sobel['outcome'] == Y)]
        boot_row = indirect_rows[
            (indirect_rows['mediator'] == med) &
            (indirect_rows['outcome'] == Y) &
            (indirect_rows['vcov_type'].str.startswith('bootstrap'))]
        if len(row) == 0:
            continue
        est = row['estimate'].values[0]
        ci_lo = boot_row['ci_lo'].values[0] if len(boot_row) > 0 else est
        ci_hi = boot_row['ci_hi'].values[0] if len(boot_row) > 0 else est
        yp = y_positions[Y] + offsets[med]
        ax.errorbar(est, yp, xerr=[[est - ci_lo], [ci_hi - est]],
                    fmt='o', color=colors[med], capsize=4, markersize=5)

ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['renew_share', 'ln_ee', 'ln_ec'])
ax.set_xlabel('Indirect effect b1×e1 (bootstrap 95% CI)')
ax.set_title('H2a/H2b: Indirect effects by mediator and outcome\n(all CIs span zero — no channel supported)')
handles = [mpatches.Patch(color=colors[m], label=labels[m]) for m in colors]
ax.legend(handles=handles, fontsize=8, loc='upper right')
plt.tight_layout()
plt.savefig(FIGS + 'h2_indirect_summary.png', dpi=150, bbox_inches='tight')
plt.show()